# cosmos_rollyourown_pose2vec — build your own pose encoder

This notebook hands you the **same 196-pose vocabulary** and the **same PI+VC training corpus**
that `cosmos_pose2vec.ipynb` uses — and then stops. Everything after Section 4 is yours to
write.

Sections 0–4 are **provided; do not edit them.** That is the whole point: every student's model
trains on byte-identical data, so results are actually comparable. Change a knob in Section 1
if you want (window radius, causal/acausal, which rats), but leave the code alone.

| Section | What it does | Yours? |
|---|---|---|
| 0 | Colab bootstrap | provided |
| 1 | Configuration — pipeline knobs only | provided (tune the knobs) |
| 2 | Pose vocabulary — the 196 "words" | provided |
| 3 | Corpus — sessions → pose sequences | provided |
| 4 | Training pairs — (centre, context) | provided |
| 5 | **What you have** — inventory of everything above | read this |
| 6 | **Guide** — how pose2vec works, and the contract your model must meet | read this |
| 7 | **What to plot** | read this |
| 8+ | **Your model, training, and figures** | **you write this** |

Reference implementation: `cosmos_pose2vec.ipynb`. Design notes:
`docs/cosmos_pose2vec_plan.md`.

## 0. Colab bootstrap

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryangrg/corner-maze-rl/blob/main/notebooks/cosmos_rollyourown_pose2vec.ipynb)

Clones the repo, installs the package, mounts Drive. No-op locally. CPU is fine — the provided
cells take a couple of seconds; how long *your* model takes is up to you.

In [ ]:
# ── Colab bootstrap (no-op locally; local-write + periodic Drive sync) ─
import sys
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    import os, subprocess, threading, atexit
    from pathlib import Path

    # ── Customise for your setup ──────────────────────────────────────
    REPO_URL          = 'https://github.com/ryangrg/corner-maze-rl.git'
    REPO_BRANCH       = 'main'
    REPO_DIR          = Path('/content/corner-maze-rl')
    USE_DRIVE         = True   # set False to skip Drive entirely
    DRIVE_ROOT        = Path('/content/drive/MyDrive/corner-maze-rl-colab')
    SYNC_INTERVAL_SEC = 120    # background rsync cadence
    # ──────────────────────────────────────────────────────────────────

    # 1. Clone repo (brings lookups + yoked dataset; both are committed)
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', '--depth=1',
                               '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        print(f'  repo already at {REPO_DIR}')

    # 2. Install package into the KERNEL's Python (sys.executable, not PATH pip —
    #    on Colab those can differ and PATH-pip installs are invisible to the kernel).
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

    # 3. Editable installs add a .pth entry, but the kernel's sys.path was built at
    #    startup and doesn't re-scan .pth files. Add the src dir manually so `import
    #    corner_maze_rl` works without a runtime restart.
    src_dir = str(REPO_DIR / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

    # 4. data/runs is ALWAYS a real local dir on the VM disk — never a symlink.
    #    This keeps writes resilient to mid-run Drive disconnects.
    local_runs = REPO_DIR / 'data' / 'runs'
    local_runs.mkdir(parents=True, exist_ok=True)

    # 5. Try Drive mount; failure just disables sync (training still works).
    drive_runs   = None
    drive_status = 'skipped (USE_DRIVE=False)'
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            if not DRIVE_ROOT.exists():
                drive_status = f'mounted, but {DRIVE_ROOT} missing — sync disabled'
            else:
                drive_runs = DRIVE_ROOT / 'runs'
                drive_runs.mkdir(exist_ok=True)
                drive_status = f'mounted at {DRIVE_ROOT}'
        except Exception as e:
            drive_status = f'mount failed ({type(e).__name__}) — sync disabled'

    # 6. Public sync helper — call from any cell with verbose=True for feedback.
    _sync_lock = threading.Lock()

    def sync_runs_to_drive(verbose: bool = False) -> tuple[bool, str]:
        """rsync local data/runs/ → Drive corner-maze-rl-colab/runs/.

        Returns (ok, message). Skips silently if Drive isn't reachable.
        Safe to call concurrently — uses a lock to prevent overlapping rsyncs.
        """
        if drive_runs is None:
            return (False, 'no drive target')
        if not _sync_lock.acquire(blocking=False):
            return (False, 'sync already in flight')
        try:
            result = subprocess.run(
                ['rsync', '-a', '--update',
                 str(local_runs) + '/', str(drive_runs) + '/'],
                capture_output=True, text=True, timeout=300,
            )
            ok  = (result.returncode == 0)
            msg = 'synced' if ok else f'rc={result.returncode}: {result.stderr.strip()[:200]}'
            if verbose:
                print(f'  [drive sync] {msg}')
            return (ok, msg)
        except Exception as e:
            if verbose:
                print(f'  [drive sync] {type(e).__name__}: {e}')
            return (False, f'{type(e).__name__}: {e}')
        finally:
            _sync_lock.release()

    # 7. Background sync — daemon thread, silent. Dies with the kernel.
    _sync_stop = threading.Event()
    def _sync_loop():
        while not _sync_stop.wait(SYNC_INTERVAL_SEC):
            sync_runs_to_drive(verbose=False)

    if drive_runs is not None:
        threading.Thread(target=_sync_loop, daemon=True).start()
        # Final sync on kernel exit so the last cycle's writes catch up.
        atexit.register(lambda: (_sync_stop.set(), sync_runs_to_drive(verbose=True)))
        sync_status = f'active (every {SYNC_INTERVAL_SEC}s + final on exit)'
    else:
        sync_status = 'inactive (runs are EPHEMERAL — download before disconnect)'

    # 8. cd into notebooks/ so REPO_ROOT = Path.cwd().parent resolves
    os.chdir(REPO_DIR / 'notebooks')

    print(f'\n✓ Colab bootstrap complete')
    print(f'  repo:    {REPO_DIR}  (branch={REPO_BRANCH})')
    print(f'  cwd:     {Path.cwd()}')
    print(f'  python:  {sys.executable}')
    print(f'  drive:   {drive_status}')
    print(f'  runs:    {local_runs}  (always local; never a symlink)')
    print(f'  sync:    {sync_status}')
    print(f'\n  manual sync any time: sync_runs_to_drive(verbose=True)')
else:
    print('✓ local environment — bootstrap skipped')

## 1. Configuration

Pipeline knobs only. There are deliberately **no model or training hyperparameters here** —
those belong to you, in your own cells further down.

`WINDOW_RADIUS` and `CAUSAL` change what counts as "nearby" in a trajectory, so they change the
training pairs. If you want your results to be comparable with the reference implementation and
with other students, leave them at `2` and `True`.

In [ ]:
from __future__ import annotations
from pathlib import Path
import torch

RUN_NAME = 'rollyourown-pose2vec'

# ── What counts as "adjacent" along a trajectory ──────────────────────
WINDOW_RADIUS = 2       # how many steps away still counts as context
CAUSAL        = True    # True  = context is the poses AFTER the centre (predict-next)
                        # False = symmetric window, WINDOW_RADIUS before AND after

# ── Which rats ────────────────────────────────────────────────────────
TRAINING_GROUP = 'pi_vc'                # 'pi' | 'pi_vc' | 'vc' | 'pi_vc_f1'

# ── Usable PI+VC subjects — all 17 are in the manuscript roster ────────
# Acquisition sessions / steps available in actions_real_pretrial:
#   CM000   7 / 12,786      CM007   9 / 11,624      CM014  10 / 18,890
#   CM001   6 / 11,462      CM008   3 /  6,827      CM015  11 / 21,253
#   CM002   7 / 12,355      CM009   7 / 16,065      CM016   5 / 11,005
#   CM003   8 / 14,069      CM010   5 / 10,951      CM017   9 / 12,346
#   CM004   8 / 20,048      CM011   5 / 10,016      CM018   9 / 13,449
#   CM005   6 / 11,801
#   CM006   3 /  7,117             all 17 pooled = 118 sessions / 222,064 steps
SUBJECTS      = 'all'   # 'all', or an explicit list e.g. ['CM004', 'CM015']
POOL_SUBJECTS = True    # True  = ONE corpus from all subjects (recommended)
                        # False = one corpus per subject

# ── Corpus ────────────────────────────────────────────────────────────
CORPUS_SOURCE       = 'real_pretrial'   # 'real_pretrial' | 'synthetic_pretrial' | 'both'
USE_CANONICAL_FRAME = True              # rotate PI/PI+VC so the cue always sits at North

SEED = 0

# ── Derived / fixed — leave alone ─────────────────────────────────────
assert WINDOW_RADIUS >= 1, 'WINDOW_RADIUS must be >= 1'

GROUP_ALIASES = {'pi': 'PI', 'pi_vc': 'PI+VC', 'vc': 'VC', 'pi_vc_f1': 'PI+VC_f1'}
assert TRAINING_GROUP in GROUP_ALIASES, f'unknown TRAINING_GROUP={TRAINING_GROUP!r}'

# 8 of the 56 yoked subjects are outside manuscript scope and must not be used for new
# training runs (README.md, "Manuscript subject roster"). None are PI+VC.
OUT_OF_MANUSCRIPT_SCOPE = {
    'CM030', 'CM032', 'CM033',            # PI
    'CM059',                              # PI+VC_f1
    'CM028', 'CM034', 'CM039', 'CM048',   # VC
}
RESTRICT_TO_MANUSCRIPT = True

FORCE_CPU = False
if FORCE_CPU:
    DEVICE = 'cpu'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

REPO_ROOT   = Path.cwd().parent
DATASET_DIR = REPO_ROOT / 'data' / 'yoked' / 'dataset'
LOOKUPS_DIR = REPO_ROOT / 'data' / 'lookups'

torch.manual_seed(SEED)

print(f'RUN_NAME  = {RUN_NAME}')
print(f'window    = radius {WINDOW_RADIUS}, {"causal" if CAUSAL else "acausal"}')
print(f'subjects  = {SUBJECTS}  (pooled={POOL_SUBJECTS})')
print(f'DEVICE    = {DEVICE}')

## 2. Pose vocabulary — the 196 "words"

The maze is a 13×13 MiniGrid, but only **49 cells** are ever occupiable: a 3×3 lattice of
corridors (45 cells) plus the 4 corner wells. With 4 headings that gives a vocabulary of
**196 poses**.

The vocabulary is derived from the environment rather than hardcoded — the env grid is this
repo's ground truth. `session_type='exposure'` is the Exposure A layout, i.e. the maze with no
barriers up, which is a strict superset of every barrier configuration used in acquisition.

**No sentinel tokens.** There is no `<START>`, `<END>`, `<PAD>`, or `<UNK>`. Skip-gram predicts
context *from* a centre, so unlike an autoregressive model it has nothing to bootstrap — the
window simply clips at the sentence edge. Emitting individual `(centre, context)` pairs means
every example is complete, so no padding is needed. And every exported row must correspond to a
real `(x, y, dir)` for the DT to index it, so sentinel rows would be junk.

In [ ]:
import warnings
import numpy as np

# pygame (pulled in by the env) emits a pkg_resources deprecation warning on import. Silence
# it so stored outputs stay clean: the stderr trace embeds a local virtualenv path, and this
# notebook is committed to a public, student-facing repo.
warnings.filterwarnings('ignore', message='pkg_resources is deprecated')

from corner_maze_rl.env.corner_maze_env import CornerMazeEnv

N_DIR     = 4
DIR_NAMES = {0: 'East', 1: 'South', 2: 'West', 3: 'North'}   # MiniGrid convention

_env = CornerMazeEnv(session_type='exposure', render_mode=None)
_env.reset(seed=0)
GRID_W, GRID_H = _env.width, _env.height


def _is_occupiable(env, x: int, y: int) -> bool:
    """A cell the agent can stand on: empty, or an object that permits overlap."""
    cell = env.grid.get(x, y)
    return cell is None or (hasattr(cell, 'can_overlap') and cell.can_overlap())


OCCUPIABLE_CELLS = sorted((x, y) for x in range(GRID_W) for y in range(GRID_H)
                          if _is_occupiable(_env, x, y))
_env.close()

# Canonical ordering: x-major, then y, then heading.
POSE_VOCAB  = [(x, y, d) for (x, y) in OCCUPIABLE_CELLS for d in range(N_DIR)]
POSE_TO_IDX = {p: i for i, p in enumerate(POSE_VOCAB)}
VOCAB_SIZE  = len(POSE_VOCAB)

assert len(OCCUPIABLE_CELLS) == 49, f'expected 49 occupiable cells, got {len(OCCUPIABLE_CELLS)}'
assert VOCAB_SIZE == 196, f'expected 196 poses, got {VOCAB_SIZE}'
assert VOCAB_SIZE == len(OCCUPIABLE_CELLS) * N_DIR

# Dense (x, y, d) -> row lookup; -1 marks "not in vocabulary" so escapes are detectable.
POSE_INDEX_GRID = np.full((GRID_W, GRID_H, N_DIR), -1, dtype=np.int32)
for _pose, _i in POSE_TO_IDX.items():
    POSE_INDEX_GRID[_pose] = _i


def rotate_pos(x, y, n):
    """n CCW rotations in screen coords (y-down): (x, y) -> (y, (W-1) - x)."""
    for _ in range(n % 4):
        x, y = y, (GRID_W - 1) - x
    return x, y


# The canonical frame rotates PI/PI+VC sessions. If the occupiable set were not closed under
# that rotation, canonicalisation could produce an out-of-vocabulary pose.
_cellset = set(OCCUPIABLE_CELLS)
for _n in (1, 2, 3):
    _img = {rotate_pos(x, y, _n) for (x, y) in OCCUPIABLE_CELLS}
    assert _img == _cellset, f'occupiable set not closed under {_n} rotation(s)'

print(f'grid            = {GRID_W} x {GRID_H}')
print(f'occupiable cells= {len(OCCUPIABLE_CELLS)}')
print(f'VOCAB_SIZE      = {VOCAB_SIZE}  ({len(OCCUPIABLE_CELLS)} cells x {N_DIR} headings)')
print(f'rotation closure= ok (1, 2, 3 CCW)')
print()
print('occupancy map  (row = y, col = x, "#" = occupiable):')
for _y in range(GRID_H):
    print('   ', ''.join('#' if (x, _y) in _cellset else '.' for x in range(GRID_W)))

## 3. Build the pose corpus — the "sentences"

One sentence per session: the rat's poses in step order. A context window must never span two
sessions, so sentences stay as a *list* of sequences and are never concatenated.

Poses come straight out of the parquet (`grid_x`, `grid_y`, `direction`) — no environment
replay is needed.

**Canonical frame.** PI and PI+VC sessions are rotated so the cue always sits at North, matching
the rat's path-integration frame. The rotation count is the cue index of the first trial. Note
the direction formula is `(d - n) % 4` — the legacy `(d + n) % 4` is wrong for odd `n` and only
accidentally right at `n = 2`.

In [ ]:
import json
import pandas as pd

_ACTION_TABLES = {
    'real_pretrial':      ['actions_real_pretrial.parquet'],
    'synthetic_pretrial': ['actions_synthetic_pretrial.parquet'],
    'both':               ['actions_real_pretrial.parquet', 'actions_synthetic_pretrial.parquet'],
}
assert CORPUS_SOURCE in _ACTION_TABLES, f'unknown CORPUS_SOURCE={CORPUS_SOURCE!r}'

subjects_df = pd.read_parquet(DATASET_DIR / 'subjects.parquet')
sessions_df = pd.read_parquet(DATASET_DIR / 'sessions.parquet')

GROUP_LABEL    = GROUP_ALIASES[TRAINING_GROUP]
_group_rows    = subjects_df[subjects_df['training_group'] == GROUP_LABEL]
_group_names   = sorted(_group_rows['subject_name'].tolist())
NAME_TO_SID    = dict(zip(_group_rows['subject_name'], _group_rows['subject_id']))

if isinstance(SUBJECTS, str) and SUBJECTS == 'all':
    SELECTED_SUBJECTS = [s for s in _group_names
                         if not (RESTRICT_TO_MANUSCRIPT and s in OUT_OF_MANUSCRIPT_SCOPE)]
    _excluded = sorted(set(_group_names) - set(SELECTED_SUBJECTS))
    if _excluded:
        print(f'  excluded {len(_excluded)} out-of-manuscript-scope subject(s): {_excluded}')
else:
    SELECTED_SUBJECTS = list(SUBJECTS)
    _unknown = set(SELECTED_SUBJECTS) - set(_group_names)
    assert not _unknown, f'not {GROUP_LABEL} subjects: {sorted(_unknown)}'
    _oos = sorted(set(SELECTED_SUBJECTS) & OUT_OF_MANUSCRIPT_SCOPE)
    if _oos:
        print(f'  [warn] explicitly selected OUT-OF-MANUSCRIPT-SCOPE subject(s): {_oos} — '
              f'not valid for published comparisons')
assert SELECTED_SUBJECTS, 'no subjects selected'

ROTATE_TO_CANONICAL = USE_CANONICAL_FRAME and GROUP_LABEL in ('PI', 'PI+VC', 'PI+VC_f1')
print(f'group={GROUP_LABEL}  subjects={len(SELECTED_SUBJECTS)}  rotate={ROTATE_TO_CANONICAL}')

# Load the action table(s) once; tag rows by source so 'both' keeps the two variants of a
# session as separate sentences instead of silently interleaving them.
_frames = []
for _fname in _ACTION_TABLES[CORPUS_SOURCE]:
    _f = pd.read_parquet(DATASET_DIR / _fname)
    _f['_src'] = _fname
    _frames.append(_f)
ACTIONS_ALL = pd.concat(_frames, ignore_index=True) if len(_frames) > 1 else _frames[0]

_required = {'session_id', 'step', 'grid_x', 'grid_y', 'direction'}
assert _required <= set(ACTIONS_ALL.columns), f'missing columns: {_required - set(ACTIONS_ALL.columns)}'


def _n_rotations_for(row) -> int | None:
    """Rotations to canonicalise a session. None => drop (cue varies within session)."""
    if not ROTATE_TO_CANONICAL:
        return 0
    tc = row.get('trial_configs')
    if tc is None or (isinstance(tc, float) and np.isnan(tc)):
        return 0
    try:
        tc = json.loads(tc) if isinstance(tc, str) else list(tc)
    except Exception:
        return 0
    if len(tc) == 0:
        return 0
    cues = {int(t[1]) for t in tc}
    if len(cues) > 1:
        return None
    return int(next(iter(cues)))


def build_sentences(subject_names) -> tuple[list[np.ndarray], pd.DataFrame]:
    """One int32 pose-index sequence per session, for the named subjects."""
    sids = {NAME_TO_SID[n] for n in subject_names}
    sess = sessions_df[sessions_df['subject_id'].isin(sids)]

    sid_to_nrot, dropped = {}, []
    for _, row in sess.iterrows():
        n = _n_rotations_for(row)
        if n is None:
            dropped.append(int(row['session_id']))
        else:
            sid_to_nrot[int(row['session_id'])] = n
    if dropped:
        print(f'  [warn] dropping {len(dropped)} multi-cue session(s): {dropped}')

    df = ACTIONS_ALL[ACTIONS_ALL['session_id'].isin(sid_to_nrot)].copy()
    if df.empty:
        return [], df

    n_rot = df['session_id'].map(sid_to_nrot).to_numpy(np.int32)
    x = df['grid_x'].to_numpy(np.int32)
    y = df['grid_y'].to_numpy(np.int32)
    d = df['direction'].to_numpy(np.int32)

    rx, ry = x.copy(), y.copy()
    for n_val in sorted(set(sid_to_nrot.values())):
        if n_val == 0:
            continue
        m = n_rot == n_val
        cx, cy = x[m], y[m]
        for _ in range(n_val % 4):
            cx, cy = cy, (GRID_W - 1) - cx
        rx[m], ry[m] = cx, cy
    rd = (d - n_rot) % 4                      # corrected formula; legacy (d + n) is wrong

    idx = POSE_INDEX_GRID[rx, ry, rd]
    n_escape = int((idx < 0).sum())
    assert n_escape == 0, (
        f'{n_escape} pose(s) outside the vocabulary — the corpus must be a subset of the '
        f'fully-open maze'
    )

    df['pose_idx'] = idx
    df = df.sort_values(['_src', 'session_id', 'step'], kind='stable')
    sentences = [g['pose_idx'].to_numpy(np.int32)
                 for _, g in df.groupby(['_src', 'session_id'], sort=False)]
    return sentences, df


CORPUS = {}
if POOL_SUBJECTS:
    CORPUS['pooled_' + TRAINING_GROUP] = SELECTED_SUBJECTS
else:
    for _s in SELECTED_SUBJECTS:
        CORPUS['subject_' + _s] = [_s]

SENTENCES = {}
for _label, _subs in CORPUS.items():
    _sents, _df = build_sentences(_subs)
    SENTENCES[_label] = _sents
    _flat  = np.concatenate(_sents) if _sents else np.empty(0, np.int32)
    _cov   = len(np.unique(_flat))
    _counts = np.bincount(_flat, minlength=VOCAB_SIZE)
    print(f'  {_label:22s} sessions={len(_sents):4d}  steps={len(_flat):7,d}  '
          f'poses covered={_cov}/{VOCAB_SIZE}')
    # Pose frequency is heavily skewed — rats dwell in some poses far more than others.
    # The rarest rows are trained on very few examples; their embeddings are the least
    # trustworthy, which matters when reading the spatial maps in the last section.
    print(f'    {"":20s} frequency  min={_counts.min():,}  p50={int(np.percentile(_counts, 50)):,}  '
          f'max={_counts.max():,}  (skew {_counts.max() / max(_counts.min(), 1):.0f}:1)')

## 4. Training pairs

Skip-gram: each centre pose is paired with each pose in its context window. Built with numpy
slicing per sentence — a Python double loop over ~222k steps would be needlessly slow.

**Session edges.** A window must never span two sessions, which is why the corpus stays a
*list* of per-session arrays and is only ever concatenated *after* pairs are formed. A pose near
the end of a session simply produces fewer pairs; nothing wraps around or bleeds into the next
session.

Clipping costs `R(R+1)/2` pairs per session when causal and `R(R+1)` when acausal, for
`R = WINDOW_RADIUS` — 3 and 6 respectively at the default `R=2`. Across 118 sessions that is
~354 of ~444,000 pairs, under 0.1%. It is a boundary effect, not a bias worth correcting.

In [ ]:
def make_pairs(sentences, radius: int, causal: bool):
    """(centre, context) index arrays. causal => context is strictly ahead of the centre."""
    offsets = list(range(1, radius + 1)) if causal else \
              [o for o in range(-radius, radius + 1) if o != 0]
    centres, contexts = [], []
    for s in sentences:
        for off in offsets:
            k = abs(off)
            if len(s) <= k:
                continue
            if off > 0:
                centres.append(s[:-k]);  contexts.append(s[k:])
            else:
                centres.append(s[k:]);   contexts.append(s[:-k])
    if not centres:
        return np.empty(0, np.int64), np.empty(0, np.int64)
    return (np.concatenate(centres).astype(np.int64),
            np.concatenate(contexts).astype(np.int64))


PAIRS, CENTRE_COVERAGE = {}, {}
for _label, _sents in SENTENCES.items():
    c, ctx = make_pairs(_sents, WINDOW_RADIUS, CAUSAL)
    PAIRS[_label] = (c, ctx)
    # Only poses that appear as a CENTRE receive gradient into input_embed — i.e. into the
    # exported encoder. Anything else is exported still at initialisation (see Section 7).
    CENTRE_COVERAGE[_label] = set(np.unique(c).tolist())
    _untrained = VOCAB_SIZE - len(CENTRE_COVERAGE[_label])
    print(f'  {_label:22s} pairs={len(c):9,d}  '
          f'({"causal" if CAUSAL else "acausal"}, radius {WINDOW_RADIUS})  '
          f'centres={len(CENTRE_COVERAGE[_label])}/{VOCAB_SIZE}'
          + (f'  [{_untrained} rows will be UNTRAINED]' if _untrained else ''))

## 5. What you have now

Everything above has run. Here is the complete inventory of what is now in memory — this is
your entire starting kit.

### The vocabulary — your 196 "words"

| Name | Type | What it is |
|---|---|---|
| `VOCAB_SIZE` | `int` | **196** — the number of distinct poses |
| `POSE_VOCAB` | `list[(x, y, dir)]` | the 196 poses in canonical order; `POSE_VOCAB[i]` is row `i` |
| `POSE_TO_IDX` | `dict[(x, y, dir) → int]` | the inverse: pose → row index |
| `POSE_INDEX_GRID` | `(13, 13, 4)` int32 | vectorised lookup; `-1` where no pose exists |
| `OCCUPIABLE_CELLS` | `list[(x, y)]` | the 49 cells the rat can stand on |
| `DIR_NAMES` | `dict` | `{0: 'East', 1: 'South', 2: 'West', 3: 'North'}` |
| `GRID_W`, `GRID_H`, `N_DIR` | `int` | 13, 13, 4 |

A **pose** is a grid position plus a heading. Poses are identified everywhere by their integer
row index in `0..195` — that integer is the only thing your model ever sees.

### The corpus — your "sentences"

| Name | Type | What it is |
|---|---|---|
| `SENTENCES` | `dict[label → list[np.ndarray]]` | one int32 array per session, in step order |
| `CORPUS` | `dict[label → list[str]]` | which subjects feed each label |
| `SELECTED_SUBJECTS` | `list[str]` | the rats actually used |

Each array is one session's pose sequence. They are kept as a **list, never concatenated** —
a context window must never span two sessions, or you would invent an adjacency between the
last pose of one session and the first of the next.

### The training pairs — your (x, y)

| Name | Type | What it is |
|---|---|---|
| `PAIRS` | `dict[label → (centres, contexts)]` | two int64 arrays of equal length |
| `CENTRE_COVERAGE` | `dict[label → set[int]]` | which poses appear as a centre |

`centres[k]` and `contexts[k]` are pose indices: pose `centres[k]` was observed within
`WINDOW_RADIUS` steps of pose `contexts[k]`. **This is the entire supervised signal.** Your job
is to predict `contexts` from `centres`.

### Helper functions still available

- `make_pairs(sentences, radius, causal)` — rebuild pairs with different settings
- `build_sentences(subject_names)` — rebuild the corpus for a different subject set
- `rotate_pos(x, y, n)` — the canonical-frame rotation

In [ ]:
# Live inventory — run this to confirm what you are starting from.
_lab = next(iter(SENTENCES))

print('VOCABULARY')
print(f'  VOCAB_SIZE       = {VOCAB_SIZE}   ({len(OCCUPIABLE_CELLS)} cells x {N_DIR} headings)')
print(f'  POSE_VOCAB[0]    = {POSE_VOCAB[0]}      (x, y, dir) -> row 0')
print(f'  POSE_VOCAB[-1]   = {POSE_VOCAB[-1]}   -> row {VOCAB_SIZE - 1}')
print(f'  POSE_INDEX_GRID  = {POSE_INDEX_GRID.shape} {POSE_INDEX_GRID.dtype}, -1 = not a pose')
print()

print('CORPUS  (sentences = one per session)')
for k, s in SENTENCES.items():
    _steps = sum(map(len, s))
    print(f'  {k:22s} {len(s):4d} sessions  {_steps:7,d} steps  '
          f'lengths {min(map(len, s))}..{max(map(len, s))}')
print()

print('TRAINING PAIRS')
for k, (c, x) in PAIRS.items():
    print(f'  {k:22s} centres {str(c.shape):>12s} {c.dtype}   '
          f'contexts {str(x.shape):>12s} {x.dtype}')
    print(f'  {"":22s} values in [{c.min()}, {c.max()}]   '
          f'centres seen: {len(CENTRE_COVERAGE[k])}/{VOCAB_SIZE}')
print()

print('EXAMPLE — the first 5 pairs, decoded')
_c, _x = PAIRS[_lab]
for k in range(5):
    print(f'  centre {_c[k]:3d} {POSE_VOCAB[_c[k]]}  ->  context {_x[k]:3d} {POSE_VOCAB[_x[k]]}')
print()

print('BASELINES to beat  (cross-entropy in nats, lower is better)')
_all = np.concatenate([x for _, x in PAIRS.values()])
_p   = np.bincount(_all, minlength=VOCAB_SIZE) / len(_all)
print(f'  uniform guess      {np.log(VOCAB_SIZE):.4f}   (know nothing)')
print(f'  marginal guess     {-(_p[_p > 0] * np.log(_p[_p > 0])).sum():.4f}   '
      f'(know only how often each pose occurs)')
print(f'  reference skip-gram ~1.32     (cosmos_pose2vec.ipynb, radius 2, causal)')

## 6. Guide — what pose2vec is, and what your model must do

### The idea

This is word2vec, with poses instead of words:

| word2vec | here |
|---|---|
| word | pose — a grid position plus a heading |
| vocabulary of V words | vocabulary of **196** poses |
| sentence | one session's pose sequence |
| "predict the neighbouring word" | "predict the neighbouring position on the grid" |
| learned word embedding | **learned pose embedding — the thing you are building** |

Train a network to predict which pose occurs near a given pose. Nobody tells it the maze
layout. But to do the task well it has to discover that some poses lead to others, and the
internal representation it invents to do that ends up encoding space. That representation is
the point — the prediction task is just the excuse.

### The task, precisely

- **Input:** a pose index, an integer in `0..195`.
- **Output:** 196 numbers — a score for every pose, saying how likely it is to be nearby.
- **Loss:** cross-entropy against the true context pose.
- **Data:** `centres` → `contexts` from `PAIRS`.

### The one-hot, and why you will not build one

The brief calls for a one-hot input: a length-196 vector, all zeros except a 1 at the pose. But
multiplying a one-hot vector by a weight matrix just **selects a row** of that matrix. So
`nn.Embedding(196, D)` *is* the one-hot input layer, computed without ever materialising the
vector. Use it. (If you would rather build the one-hot explicitly to see it work, that is a
perfectly good first version — just expect it to be slower.)

### The contract

Whatever you build, give it these two methods so the plotting and comparison work:

```python
class MyPoseModel(nn.Module):
    def forward(self, centre_idx):        # (B,) int64  ->  (B, 196) logits
        ...

    def pose_representations(self):       # -> (196, D) numpy, one row per pose
        ...
```

`D` is yours to choose. **196 is the number of poses, not the width** — it is always the row
count. `D` is how many numbers describe each pose, and it should be well under 196: the
compression is what forces poses to share dimensions, and that sharing is what produces
spatial structure. A width of 196 lets every pose have a private dimension and learn nothing
general.

### A training loop, in outline

```python
model = MyPoseModel(...).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()

centres, contexts = PAIRS['pooled_pi_vc']
ds = TensorDataset(torch.from_numpy(centres), torch.from_numpy(contexts))
dl = DataLoader(ds, batch_size=4096, shuffle=True)

for epoch in range(EPOCHS):
    for cb, xb in dl:
        opt.zero_grad()
        loss = lossf(model(cb.to(DEVICE)), xb.to(DEVICE))
        loss.backward()
        opt.step()
```

A full softmax over 196 classes is cheap — skip negative sampling and hierarchical softmax,
which exist for vocabularies of 100,000+ and buy you nothing here.

### Ideas worth trying

- **Vary the width.** Where does spatial structure appear and where does it collapse?
- **Add a hidden layer.** Does a non-linearity help, or does the linear version already win?
- **Causal vs acausal.** Does removing the arrow of time change what is learned?
- **Vary `WINDOW_RADIUS`.** Wider context should mean coarser, more place-like fields.
- **CBOW instead of skip-gram** — predict the centre from the context. Note you will need
  padding for variable-length context.
- **Hold out data.** Split by *session* (not by pair — pairs from the same session overlap and
  would leak). Then a lower validation loss actually means a better model.

## 7. What to plot

### The main figure: spatial tuning, four panels

For a chosen unit of your representation, draw **four heatmaps side by side, one per heading**
— East, South, West, North. Panel `d`, at cell `(x, y)`, shows

```python
value = representations[POSE_TO_IDX[(x, y, d)], unit]
```

Build each panel as a `(GRID_H, GRID_W)` array pre-filled with `np.nan`, scatter the 196 values
in, and `imshow` it. The `NaN`s leave non-occupiable cells as background, so the maze's
corridor lattice appears — which is also a free check that your indexing is right. If the maze
shape does not appear, your pose→row mapping is wrong.

Rules that matter:

- **Share one colour scale across all four panels**, symmetric about zero
  (`vmin=-vmax, vmax=+vmax`), with a diverging colormap. Otherwise the panels are not
  comparable and you will read differences that are not there.
- **`y` points down.** Set `ax.set_ylim(GRID_H - 0.5, -0.5)` with `origin='upper'`.
- **Label the axes and mark a landmark.** The 49-cell lattice is symmetric under transpose and
  under 90° rotation, so a transposed or rotated plot still *looks* like a maze. Tick labels
  plus a marker on the North cue at `(6, 1)` are what make the orientation checkable.

A maze outline to overlay, if you want one:

```python
import matplotlib.patches as mpatches
from corner_maze_rl.env.constants import WELL_LOCATIONS, CUE_LOCATIONS

def draw_maze(ax):
    for (x, y) in OCCUPIABLE_CELLS:
        ax.add_patch(mpatches.Rectangle((x - .5, y - .5), 1, 1,
                                        facecolor='none', edgecolor='0.75', lw=.4))
    for wx, wy in WELL_LOCATIONS:                      # the four corner wells
        ax.add_patch(mpatches.Rectangle((wx - .4, wy - .4), .8, .8,
                                        facecolor='none', edgecolor='steelblue', lw=1.5))
    ax.add_patch(mpatches.Circle((6, 1), .3, facecolor='gold', edgecolor='black'))  # N cue
    ax.set_xlim(-.5, GRID_W - .5); ax.set_ylim(GRID_H - .5, -.5)
    ax.set_aspect('equal')
```

### How to read it

- **A localised blob** = a place field: that unit fires in one region.
- **Different panels, different pattern** = heading-tuned: the unit cares which way the rat
  faces, not just where it is.
- **All four panels identical** = the unit encodes position and ignores heading.
- **Washed-out, no structure** = that unit is not doing much. Rank units by the standard
  deviation of their column and look at the top ones first — most units are boring.

### Also worth plotting

- **The loss curve.** Compare against the baselines printed in Section 5. If you are not well
  under the marginal-guess baseline, the model has learned essentially nothing.
- **Your model vs the reference.** Load `data/lookups/pose2vec_*.npz` (`keys` gives the
  `(x, y, dir)` per row, `vectors` the representation) and plot a unit beside one of yours.
- **A similarity map.** Pick a pose, compute the distance from its vector to every other pose's,
  and plot that as the same four-panel figure. A good embedding should light up the poses
  physically near it — this is often more convincing than any single unit.

## 8. Your turn

Everything below is yours. Suggested order:

1. Define your model — satisfy the contract in Section 6.
2. Train it, printing loss per epoch.
3. Compare your final loss against the Section 5 baselines.
4. Plot the four-panel spatial tuning for your most strongly tuned units.
5. Write down what you see, and what you expected to see.

In [ ]:
# ── Your model goes here ──────────────────────────────────────────────
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


class MyPoseModel(nn.Module):
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        raise NotImplementedError('build me')

    def forward(self, centre_idx: torch.Tensor) -> torch.Tensor:
        """(B,) int64 pose indices -> (B, 196) logits."""
        raise NotImplementedError

    def pose_representations(self) -> 'np.ndarray':
        """(196, D) — one row per pose, in POSE_VOCAB order."""
        raise NotImplementedError